# Week 23, Day 2: Hopf bifurcations and limit cycles

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w023/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Amplitude grows as the square root

Simulate ż = (μ + iω)z − |z|²z in the complex plane with ω = 2 (RK4, dt = 0.05, 20,000 steps from z = 0.01) for μ = 0.01, 0.04, 0.16 and 0.64. Record the amplitude (the mean |z| over the last 1,000 steps) in `amps`, and the frequency in `freqs`, from the average change of angle per unit time over the same steps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
mus, omega = np.array([0.01, 0.04, 0.16, 0.64]), 2.0
amps, freqs = [], []
for mu in mus:
    f = lambda z: (mu + 1j * omega) * z - abs(z) ** 2 * z
    z, dt, zs = 0.01 + 0j, 0.05, []
    for _ in range(20000):
        k1 = f(z); k2 = f(z + dt / 2 * k1); k3 = f(z + dt / 2 * k2); k4 = f(z + dt * k3)
        z = z + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4); zs.append(z)
    zs = np.array(zs[-1000:])
    amps.append(...)
    freqs.append(np.mean(np.diff(np.unwrap(np.angle(zs)))) / dt / (2 * np.pi))
amps, freqs = np.array(amps), np.array(freqs)
print("amplitudes", amps.round(4), " sqrt(mu)", np.sqrt(mus).round(4), ";  frequencies", freqs.round(4))
plt.loglog(mus, amps, "o-"); plt.xlabel("mu"); plt.ylabel("amplitude"); plt.show()

In [ ]:
import numpy as np
def test_amp():
    "the measured amplitudes are within 1% of sqrt(mu)"
    assert np.all(np.abs(amps - np.sqrt(mus)) / np.sqrt(mus) < 0.01)

def test_freq():
    "the oscillation frequency stays at omega / 2 pi within 1% for every mu"
    assert np.all(np.abs(freqs - 2 / (2 * np.pi)) / (2 / (2 * np.pi)) < 0.01)

test_amp()
test_freq()
print("All checks passed")

### Exercise: Two outcomes from one parameter

For μ = −0.1 compute the radii of the unstable and stable cycles, `r_small` and `r_big`, from the formula. Then integrate the radius equation (Euler, dt = 0.01, 10,000 steps) from ρ = 0.2 and from ρ = 1.0, storing the final radii in `final[start]`.

In [ ]:
mu = -0.1
r_small = ...
r_big = ...
final = {}
for r0 in (0.2, 1.0):
    rho = r0
    for _ in range(10000):
        rho += 0.01 * (mu * rho + rho ** 3 - rho ** 5)
    final[r0] = rho
print(f"unstable cycle at {r_small:.4f}, stable cycle at {r_big:.4f};  final radii {({k: round(v, 4) for k, v in final.items()})}")

In [ ]:
import numpy as np
def test_two():
    "at mu = -0.1, a start at 0.2 decays to below 1e-3 and a start at 1.0 settles within 1e-3 of the large cycle"
    assert final[0.2] < 1e-3 and abs(final[1.0] - r_big) < 1e-3

def test_threshold():
    "the unstable cycle separates them: r_small lies between 0.2 and 1.0"
    assert 0.2 < r_small < 1.0

test_two()
test_threshold()
print("All checks passed")

In [ ]:
gNa, gK, gL, ENa, EK, EL, C = 120.0, 36.0, 0.3, 50.0, -77.0, -54.387, 1.0
def hh_rates(V):
    an = 0.01 * (V + 55) / (1 - np.exp(-(V + 55) / 10)); bn = 0.125 * np.exp(-(V + 65) / 80)
    am = 0.1 * (V + 40) / (1 - np.exp(-(V + 40) / 10)); bm = 4.0 * np.exp(-(V + 65) / 18)
    ah = 0.07 * np.exp(-(V + 65) / 20); bh = 1 / (1 + np.exp(-(V + 35) / 10))
    return an, bn, am, bm, ah, bh
def hh_rhs(y, I):
    V, n, m, h = y
    an, bn, am, bm, ah, bh = hh_rates(V)
    return np.array([(I - gNa * m ** 3 * h * (V - ENa) - gK * n ** 4 * (V - EK) - gL * (V - EL)) / C,
                     an * (1 - n) - bn * n, am * (1 - m) - bm * m, ah * (1 - h) - bh * h])
def steady(V):
    an, bn, am, bm, ah, bh = hh_rates(V)
    return np.array([V, an / (an + bn), am / (am + bm), ah / (ah + bh)])
def hh_rest(I):
    g = lambda V: hh_rhs(steady(V), I)[0]; lo, hi = -75.0, -50.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if g(lo) * g(mid) <= 0: hi = mid
        else: lo = mid
    return steady((lo + hi) / 2)
count_spikes = lambda V: int(np.sum((V[1:] > 0) & (V[:-1] <= 0)))
print("rest at I = 8:", hh_rest(8.0).round(4))

### Exercise: Rest and firing, side by side

Write `simulate(Ifun, y, T)` (RK4, dt = 0.01 ms, returning the voltage at every step). Starting from the resting state for I = 8, simulate 100 ms with constant input 8 (`V_quiet`), and with an extra 20 μA/cm² from 10 to 11 ms (`V_pulse`).

In [ ]:
def simulate(Ifun, y, T, dt=0.01):
    Vs = np.empty(int(T / dt))
    for i in range(len(Vs)):
        t = i * dt
        k1 = hh_rhs(y, Ifun(t)); k2 = hh_rhs(y + dt / 2 * k1, Ifun(t + dt / 2))
        k3 = hh_rhs(y + dt / 2 * k2, Ifun(t + dt / 2)); k4 = hh_rhs(y + dt * k3, Ifun(t + dt))
        y = ...
        Vs[i] = y[0]
    return Vs

y8 = hh_rest(8.0)
V_quiet = simulate(lambda t: 8.0, y8, 100.0)
V_pulse = ...
print(f"no pulse: {count_spikes(V_quiet)} spikes;  with pulse: {count_spikes(V_pulse)} spikes, {count_spikes(V_pulse[5000:])} in the last 50 ms")
ts = np.arange(len(V_pulse)) * 0.01
plt.plot(ts, V_quiet, label="no pulse"); plt.plot(ts, V_pulse, label="pulse at 10 ms"); plt.xlabel("ms"); plt.ylabel("V (mV)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_rest():
    "without the pulse the neuron stays at rest at I = 8 (voltage range below 0.01 mV)"
    assert np.ptp(V_quiet) < 0.01

def test_fire():
    "after the pulse it keeps firing: at least 3 spikes in the last 50 ms"
    assert count_spikes(V_pulse[5000:]) >= 3

test_rest()
test_fire()
print("All checks passed")